In [1]:
import numpy as np

def power_turb(u_wind, a_rotor, rho_air, delta_u_s, qs):
    # Function to calculate Turbine power
    c_p = 0.5 * (1 + qs) * (1 - qs**2)  # Power Coefficient
    v = u_wind * (1 - np.sqrt(delta_u_s))  # Wind speed after the rotor disc
    P = 0.5 * (rho_air * a_rotor * v**3 * c_p)  # Turbine Power
    return v, P

In [2]:
def velocity_def(n_turbine, m_turbine, r_rotor, k, xs, qs, ovl):
    # Function to calculate the velocity deficit
    nm = n_turbine * m_turbine
    delta_u_s = np.zeros((nm, 1))

    for i in range(nm):
        for j in range(nm):
            delta_u_s[i] += ovl[i, j] * ((1 - qs[j]) / (1 + k * np.abs(xs[i] - xs[j]) / r_rotor)**2)**2

    return delta_u_s

In [3]:
def area_ov(rw, ydis, R_rotor):
    # Function to calculate the overlap area
    t1 = rw**2 * np.arccos((rw**2 - R_rotor**2 + ydis**2) / (2 * ydis * rw))
    t2 = R_rotor**2 * np.arccos((R_rotor**2 - rw**2 + ydis**2) / (2 * ydis * R_rotor))
    t3 = ((rw + R_rotor)**2 - ydis**2) * (ydis**2 - (rw - R_rotor)**2)
    ovl = t1 + t2 - 0.5 * np.sqrt(t3)
    return ovl

def overlap(n_turbine, m_turbine, R_rotor, k, xs, ys):
    # Function to calculate the overlap area
    nm = n_turbine * m_turbine
    ovl = np.zeros((nm, nm))

    for i in range(nm):
        for j in range(nm):
            xdis = xs[i] - xs[j]
            if xdis > 0:
                rw = R_rotor + (k * np.abs(xdis))
                rlap = rw + R_rotor
                fullap = rw - R_rotor

                ydis = np.abs(ys[i] - ys[j])
                if fullap < ydis < rlap:
                    ovl[i, j] = area_ov(rw, ydis, R_rotor)
                elif ydis <= fullap:
                    ovl[i, j] = np.pi * R_rotor**2
                else:
                    ovl[i, j] = 0
            else:
                ovl[i, j] = 0

    ovl /= np.pi * R_rotor**2
    return ovl


In [4]:
def wind_farm(angl, n_turbine, m_turbine):
    # Rotor disc radius
    R_rotor = 41.2  # [m]
    dx = 481.70637862320414186300933810117
    dy = 858.56241559894146317763280998799
    xR = dx / R_rotor
    yR = dy / R_rotor
    x0 = 0
    y0 = 0
    x = np.ones((n_turbine, m_turbine))
    y = np.ones((n_turbine, m_turbine))

    for i in range(n_turbine):
        for j in range(m_turbine):
            x[i, j] = (i - 1) * dx + np.tan(np.deg2rad(8)) * (j - 1) * dy
            y[i, j] = ((j - 1) * dy) + np.tan(np.deg2rad(2)) * (i - 1) * dx

    if angl != 0:
        anglr = -np.deg2rad(angl)
        rx = np.zeros_like(x)
        ry = np.zeros_like(y)

        for i in range(n_turbine):
            for j in range(m_turbine):
                rx[i, j] = np.cos(anglr) * x[i, j] + np.sin(anglr) * y[i, j]
                ry[i, j] = -np.sin(anglr) * x[i, j] + np.cos(anglr) * y[i, j]
                x[i, j] = rx[i, j]
                y[i, j] = ry[i, j]

    return x, y